# 05 - Generate Load Balancer Statistics

This notebook aggregates historical data from the database and generates the `per_task_model_stats.json` file required by the Load Balancer's `StatsRegistry`.

**Output Path**: `artemis_final/ares/aggregates/per_task_model_stats.json`

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys
from pathlib import Path
import json
import logging
import pandas as pd
from sqlalchemy import text


In [7]:
# Path Setup
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent.parent
if str(ARTEMIS_DIR) not in sys.path:
    sys.path.append(str(ARTEMIS_DIR))
ARTEMIS_DIR

PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router')

In [8]:

from artemis_final.ares.db.connection import get_engine
from artemis_final.common.config_loader import STATS_PATH

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)-15s | %(message)s')
logger = logging.getLogger("stats_gen")

In [9]:
# Connect to Database
engine = get_engine()
print(f"Connected to DB. Stats will be saved to: {STATS_PATH}")

Connected to DB. Stats will be saved to: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/ares/aggregates/per_task_model_stats.json


In [ ]:
# Query Data

# We only need:
#  - router_task
#  - model_name
#  - latency_ms
#  - is_correct     (0/1 accuracy signal)
#  - estimated_cost_usd
# Optionally: judge_molmo_score for debugging, but we MUST NOT filter on it.

query = """
SELECT 
    s.router_task,
    r.model_name,
    r.latency_ms,
    r.is_correct,
    r.estimated_cost_usd,
    e.judge_molmo_score  -- optional, may be NULL
FROM vlm_responses r
JOIN vlm_samples s 
    ON r.sample_id = s.sample_id
LEFT JOIN vlm_evaluations e 
    ON r.sample_id = e.sample_id 
   AND r.model_name = e.model_name
WHERE r.ok = true
  AND s.router_task IS NOT NULL
  AND r.model_name IS NOT NULL
"""

print("Loading data...")
df = pd.read_sql(query, engine)
print(f"Loaded {len(df)} rows.")
display(df.head())


Loading data...
Loaded 275375 rows.


,router_task,model_name,latency_ms,is_correct,judge_molmo_score,total_tokens,estimated_cost_usd
0,web_understanding,qwen3_vl_8b_thinking,21005.0,False,10.0,5015,0.005111
1,knowledge_vqa,qwen2_5_vl_3b,1641.0,False,10.0,449,0.000049
2,document_ocr,qwen2_5_vl_7b,847.0,True,10.0,4952,0.000990
3,icon_reasoning,qwen3_vl_8b_thinking,2990.0,False,10.0,234,0.000249
4,document_ocr,qwen2_5_vl_3b,1665.0,True,10.0,4951,0.000495


In [ ]:
# Aggregate Statistics

import numpy as np

stats = {}

if not df.empty:
    tasks = df["router_task"].unique()
    print(f"Processing {len(tasks)} tasks: {tasks}")

    for task in tasks:
        task_df = df[df["router_task"] == task]
        stats[task] = {}

        models = task_df["model_name"].unique()
        for model in models:
            m_df = task_df[task_df["model_name"] == model]

            if m_df.empty:
                continue

            # --- Latency metrics ---
            latencies = m_df["latency_ms"].dropna()
            if not latencies.empty:
                avg_latency = float(latencies.mean())
                p50_latency = float(latencies.median())
                p95_latency = float(latencies.quantile(0.95))
            else:
                avg_latency = p50_latency = p95_latency = float("nan")

            # --- Accuracy ---
            # is_correct should be 0/1 or bool; we cast to float and take mean
            if "is_correct" in m_df.columns and m_df["is_correct"].notnull().any():
                acc_series = m_df["is_correct"].astype(float)
                avg_accuracy = float(acc_series.mean())
            else:
                avg_accuracy = float("nan")

            # --- Cost per request ---
            if "estimated_cost_usd" in m_df.columns and m_df["estimated_cost_usd"].notnull().any():
                cost_per_request = float(m_df["estimated_cost_usd"].mean())
            else:
                cost_per_request = float("nan")

            # For reporting / LB heuristics:
            cost_per_1000 = cost_per_request * 1000.0 if np.isfinite(cost_per_request) else float("nan")

            # --- Sample count (for weighting later) ---
            num_samples = int(len(m_df))

            stats[task][model] = {
                "num_samples": num_samples,
                "avg_latency_ms": avg_latency,
                "p50_latency_ms": p50_latency,
                "p95_latency_ms": p95_latency,
                "avg_accuracy": avg_accuracy,
                "cost_per_request_usd": cost_per_request,
                "cost_per_1000_requests_usd": cost_per_1000,
            }
else:
    print("Warning: DataFrame is empty!")

# Display sample
import json
print(json.dumps(stats, indent=2)[:500] + "...")


Processing 30 tasks: ['web_understanding' 'knowledge_vqa' 'document_ocr' 'icon_reasoning'
 'chart_reasoning' 'meme_classification' 'general_vqa' 'code_generation'
 'chart_captioning' 'table_math' 'geometry_reasoning' 'diagram_reasoning'
 'map_reasoning' 'handwriting_ocr' 'spatial_reasoning' 'table_reasoning'
 'dense_captioning' 'medical_report' 'textbook_qa' 'diagram_captioning'
 'scene_text_ocr' 'abstract_reasoning' 'rendered_text_ocr'
 'science_reasoning' 'ui_captioning' 'difference_detection' 'counting'
 'visual_mrc' 'image_captioning' 'medical_vqa']
{
  "web_understanding": {
    "qwen3_vl_8b_thinking": {
      "avg_latency_ms": 21981.712672521957,
      "avg_accuracy": 0.7693851944792974,
      "cost_per_request_usd": 0.0047764841405269765
    },
    "gemma_3_27b": {
      "avg_latency_ms": 24006.603213844253,
      "avg_accuracy": 0.6093943139678616,
      "cost_per_request_usd": 0.0002069688133498146
    },
    "qwen2_5_vl_3b": {
      "avg_latency_ms": 6442.338832487309,
      

In [12]:
# Save to JSON
STATS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
    
print(f"\u2705 Statistics saved to {STATS_PATH}")

✅ Statistics saved to /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/ares/aggregates/per_task_model_stats.json
